# gLM2 Genomic Language Model

![gLM2](https://proto-bio.github.io/proto-assets/images/tool/glm2/hero.png)

This notebook demonstrates the four functions of the gLM2 tool. In the first section, we use `run_glm2_embeddings` to extract a locus embedding and per-token logits. In the second section, we use `run_glm2_score` to compute masked pseudo-log-likelihood metrics that measure how natural a locus appears to the model. In the third section, we use `run_glm2_sample` to resample masked positions while keeping each position's protein or DNA identity. In the fourth section, we use `run_glm2_gradient` to compute gradients of the masked log-likelihood with respect to a relaxed sequence. Unlike protein-only language models, gLM2 reads a genomic locus as a mix of protein-coding genes and intergenic DNA, so its representations capture context from neighboring genes and the regulatory sequence between them.

A locus is written with uppercase amino acids for each gene, lowercase `acgt` for intergenic DNA, and a `+` or `-` strand marker before each element.

In [ ]:
from pathlib import Path

from proto_tools.tools.masked_models.glm2 import (
    GLM2EmbeddingsConfig, GLM2EmbeddingsInput, run_glm2_embeddings,
    GLM2ScoringConfig, GLM2ScoringInput, run_glm2_score,
    GLM2SampleConfig, GLM2SampleInput, run_glm2_sample,
    GLM2GradientConfig, GLM2GradientInput, run_glm2_gradient,
)
from proto_tools.tools.masked_models.mixed_data_models import one_hot_mixed_logits
from proto_tools.transforms.masking import MaskingStrategy
from proto_tools.utils.notebook_docs import display_api_reference, display_available_tools

CHECKPOINT = "tattabio/gLM2_150M"
DEVICE = "cuda"

display_available_tools("glm2")

## Prepared example locus

The protein and DNA segments below are shortened from the mixed-locus example in the [gLM2 upstream README](https://github.com/TattaBio/gLM2#usage). This illustrates the input notation.

The smaller fragment is used for the more expensive one-mask-per-position score and gradient calculations.

In [ ]:
SEQUENCE = "+MALTKVEKRNRIK+aatttaaggaa-MLGIDNIERVKP"
SHORT_SEQUENCE = "+MALTKV+aatt-MLGIDN"
print(SEQUENCE)

## Embeddings and logits

The pooled vector averages all unpadded tokens, including strand markers. Optional raw logits have one row per token and 24 columns in `ACDEFGHIKLMNPQRSTVWYacgt` order.

In [ ]:
display_api_reference("glm2-embedding", "input", "run_glm2_embeddings")
display_api_reference("glm2-embedding", "config", "run_glm2_embeddings")
display_api_reference("glm2-embedding", "output", "run_glm2_embeddings")

In [ ]:
embeddings_result = run_glm2_embeddings(
    GLM2EmbeddingsInput(sequences=[SEQUENCE]),
    GLM2EmbeddingsConfig(
        model_checkpoint=CHECKPOINT, device=DEVICE, return_logits=True,
    ),
)
assert embeddings_result.success, embeddings_result.errors
embedding = embeddings_result.results[0]
print("Pooled dimension:", len(embedding.mean_embedding))
print("Token count:", len(embedding.logits))
print("Logit columns:", embedding.vocab)
print("First tokens:", list(enumerate(SEQUENCE[:8], start=1)))

## Masked pseudo-log-likelihood

Every canonical protein/DNA position is masked in turn and scored from its context. Markers and ambiguous protein context are excluded from the score targets. Higher mean log-likelihood and lower perplexity indicate greater model predictability; they do not establish function.

In [ ]:
display_api_reference("glm2-score", "input", "run_glm2_score")
display_api_reference("glm2-score", "config", "run_glm2_score")
display_api_reference("glm2-score", "output", "run_glm2_score")

In [ ]:
score_result = run_glm2_score(
    GLM2ScoringInput(sequences=[SHORT_SEQUENCE]),
    GLM2ScoringConfig(
        model_checkpoint=CHECKPOINT, device=DEVICE, batch_size=1, return_logits=True,
    ),
)
assert score_result.success, score_result.errors
score = score_result.scores[0]
print("Mean log-likelihood:", score["avg_log_likelihood"])
print("Perplexity:", score["perplexity"])
print("Scored token positions:", score.scored_positions)

## Sampling while preserving modality

The automatic strategy selects two canonical biological positions. `fixed_positions=[2]` protects the first amino acid because the initial strand marker occupies position 1. The model may sample the original token, so two selected sites need not mean two substitutions.

In [ ]:
display_api_reference("glm2-sample", "input", "run_glm2_sample")
display_api_reference("glm2-sample", "config", "run_glm2_sample")
display_api_reference("glm2-sample", "output", "run_glm2_sample")

In [ ]:
sample_result = run_glm2_sample(
    GLM2SampleInput(sequences=[SEQUENCE]),
    GLM2SampleConfig(
        model_checkpoint=CHECKPOINT,
        device=DEVICE,
        masking_strategy=MaskingStrategy(num_mutations=2, fixed_positions=[2]),
        temperature=1.0,
        seed=7,
    ),
)
assert sample_result.success, sample_result.errors
sample = sample_result.results[0]
assert len(sample.sequence) == len(SEQUENCE)
assert sample.sequence[1] == SEQUENCE[1]
print("Original:", SEQUENCE)
print("Sampled: ", sample.sequence)

### Explicit masks with a protein/DNA label per site

An underscore does not carry case, so each premasked position needs an explicit modality. The mapping uses 1-indexed token positions. This example fills one protein and one DNA site through four refinement rounds. Explicit masks bypass automatic position selection.

In [ ]:
masked_tokens = list(SEQUENCE)
protein_position = 2
dna_position = next(i for i, token in enumerate(masked_tokens, start=1) if token in "acgt")
masked_tokens[protein_position - 1] = "_"
masked_tokens[dna_position - 1] = "_"

premasked_result = run_glm2_sample(
    GLM2SampleInput(
        sequences=["".join(masked_tokens)],
        mask_modalities=[{protein_position: "protein", dna_position: "dna"}],
    ),
    GLM2SampleConfig(
        model_checkpoint=CHECKPOINT,
        device=DEVICE,
        sampling_method="iterative_refinement",
        num_steps=4,
        top_p=0.9,
        temperature_annealing=False,
        seed=7,
    ),
)
assert premasked_result.success, premasked_result.errors
completed = premasked_result.results[0]
assert completed.sequence[protein_position - 1] in "ACDEFGHIKLMNPQRSTVWY"
assert completed.sequence[dna_position - 1] in "acgt"
print(completed.sequence)

## Relaxed-sequence gradient

The template fixes modality and strand markers. `one_hot_mixed_logits()` creates one 24-column row per token, with zero rows for fixed context. The default input temperature applies softmax within the allowed alphabet; use `temperature=None` only for already normalized modality-restricted probabilities. The model weights remain frozen.

In [ ]:
display_api_reference("glm2-gradient", "input", "run_glm2_gradient")
display_api_reference("glm2-gradient", "config", "run_glm2_gradient")
display_api_reference("glm2-gradient", "output", "run_glm2_gradient")

In [ ]:
gradient_result = run_glm2_gradient(
    GLM2GradientInput(
        inputs={"sequence": SHORT_SEQUENCE, "logits": one_hot_mixed_logits(SHORT_SEQUENCE, sharpness=2.0)},
        temperature=1.0,
    ),
    GLM2GradientConfig(model_checkpoint=CHECKPOINT, device=DEVICE, batch_size=1),
)
assert gradient_result.success, gradient_result.errors
candidate = gradient_result.results[0]  # one result per candidate; vocab stays on the output
assert candidate.gradient is not None
print("Masked mean NLL:", candidate.loss)
print("Gradient shape:", (len(candidate.gradient), len(gradient_result.vocab)))
for token, row in zip(SHORT_SEQUENCE, candidate.gradient, strict=True):
    if token in ("+", "-"):
        assert all(value == 0.0 for value in row)

## Export

Exports are written to `example_output/` next to this notebook. Embedding CSV contains pooled vectors; score, sample, and gradient JSON preserve their result metadata.

In [ ]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

embeddings_result.export(name="glm2_embeddings", export_path=output_dir, file_format="csv")
score_result.export(name="glm2_scores", export_path=output_dir, file_format="json")
sample_result.export(name="glm2_samples", export_path=output_dir, file_format="json")
gradient_result.export(name="glm2_gradient", export_path=output_dir, file_format="json")
print("Written files:", sorted(path.name for path in output_dir.iterdir()))